# 05 Model Evaluation

This notebook compares the baseline and deep learning ADR models using the artifacts saved by `03_baseline_models.ipynb` and `04_deep_learning_models.ipynb`.

Run the training notebooks first so the expected files exist under `results/`.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Allow the notebook to run whether Jupyter was launched from the repo root
# or from inside the notebooks/ directory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'results').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Results dir: {RESULTS_DIR}')

In [ ]:
# These two CSV files are the top-level summary outputs from notebooks 03 and 04.
baseline_metrics_path = RESULTS_DIR / 'baseline_models_metrics.csv'
deep_metrics_path = RESULTS_DIR / 'deep_models_metrics.csv'

missing_files = [
    str(path.relative_to(PROJECT_ROOT))
    for path in [baseline_metrics_path, deep_metrics_path]
    if not path.exists()
]
if missing_files:
    raise FileNotFoundError(
        'Missing required results artifacts. Run the training notebooks first: ' + ', '.join(missing_files)
    )

# Baseline notebook does not save a threshold for every model, so keep that
# column present and fill with NaN for schema consistency before concatenation.
baseline_df = pd.read_csv(baseline_metrics_path).copy()
baseline_df['Family'] = 'Baseline ML'
baseline_df['Selected Threshold'] = np.nan

# Deep model notebook already includes selected threshold values.
deep_df = pd.read_csv(deep_metrics_path).copy()
deep_df['Family'] = 'Deep Learning'

# Combine both families into one table so every model can be ranked together.
combined_metrics = pd.concat([baseline_df, deep_df], ignore_index=True, sort=False)
combined_metrics = combined_metrics[
    ['Family', 'Model', 'Val AUROC', 'Val AUPRC', 'Test AUROC', 'Test AUPRC', 'Selected Threshold']
]
combined_metrics = combined_metrics.sort_values(['Test AUROC', 'Test AUPRC'], ascending=False).reset_index(drop=True)

display(combined_metrics.style.format({
    'Val AUROC': '{:.4f}',
    'Val AUPRC': '{:.4f}',
    'Test AUROC': '{:.4f}',
    'Test AUPRC': '{:.4f}',
    'Selected Threshold': '{:.2f}'
}))

# Save a flat summary table that can be reused in reports or slides.
combined_metrics.to_csv(RESULTS_DIR / 'combined_model_metrics.csv', index=False)
print('Saved combined metrics to results/combined_model_metrics.csv')

In [ ]:
# Visualize the two main ranking metrics side by side to show whether the same
# models stay strong under both AUROC and AUPRC.
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Keep a consistent color mapping so baseline vs deep learning is easy to spot.
plot_df = combined_metrics.sort_values('Test AUROC', ascending=False)
palette = {'Baseline ML': '#4C78A8', 'Deep Learning': '#F58518'}

sns.barplot(data=plot_df, x='Test AUROC', y='Model', hue='Family', dodge=False, palette=palette, ax=axes[0])
axes[0].set_title('Test AUROC by Model')
axes[0].set_xlabel('Test AUROC')
axes[0].set_ylabel('Model')
axes[0].legend(title='Family', loc='lower right')

sns.barplot(data=plot_df.sort_values('Test AUPRC', ascending=False), x='Test AUPRC', y='Model', hue='Family', dodge=False, palette=palette, ax=axes[1])
axes[1].set_title('Test AUPRC by Model')
axes[1].set_xlabel('Test AUPRC')
axes[1].set_ylabel('Model')
axes[1].legend_.remove()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'model_metric_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# This scatter plot makes the tradeoff between AUROC and AUPRC visible in one view.
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=combined_metrics,
    x='Test AUROC',
    y='Test AUPRC',
    hue='Family',
    style='Family',
    s=120,
    palette=palette
)
for _, row in combined_metrics.iterrows():
    plt.text(row['Test AUROC'] + 0.001, row['Test AUPRC'] + 0.001, row['Model'], fontsize=9)

plt.title('Test-Set Ranking Metrics')
plt.xlabel('Test AUROC')
plt.ylabel('Test AUPRC')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'model_metric_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Convert saved predictions into a common threshold-based metrics row so the
# best baseline model and all deep models can be compared in one table.
def summarize_predictions(model_name, family, y_true, y_pred, threshold):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    return {
        'Family': family,
        'Model': model_name,
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'Specificity': specificity,
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'TP': int(tp),
        'TN': int(tn),
        'FP': int(fp),
        'FN': int(fn),
    }


prediction_summaries = []

# Notebook 03 currently saves row-level predictions only for the selected best
# baseline model, so threshold analysis for baselines is limited to that model.
best_baseline_predictions_path = RESULTS_DIR / 'best_baseline_test_predictions.csv'
best_baseline_report_path = RESULTS_DIR / 'best_baseline_classification_report.json'
if best_baseline_predictions_path.exists() and best_baseline_report_path.exists():
    baseline_preds = pd.read_csv(best_baseline_predictions_path)
    with open(best_baseline_report_path, 'r') as f:
        baseline_meta = json.load(f)

    prediction_summaries.append(
        summarize_predictions(
            baseline_meta['best_model'],
            'Baseline ML',
            baseline_preds['y_true'],
            baseline_preds['y_pred_tuned'],
            baseline_meta['selected_threshold'],
        )
    )

# Notebook 04 saves per-model predictions for every deep learning model.
deep_predictions_path = RESULTS_DIR / 'deep_models_test_predictions.json'
deep_metrics_json_path = RESULTS_DIR / 'deep_models_metrics.json'
if deep_predictions_path.exists() and deep_metrics_json_path.exists():
    with open(deep_predictions_path, 'r') as f:
        deep_predictions = json.load(f)
    with open(deep_metrics_json_path, 'r') as f:
        deep_metrics = json.load(f)

    for model_name, payload in deep_predictions.items():
        prediction_summaries.append(
            summarize_predictions(
                model_name,
                'Deep Learning',
                np.array(payload['y_true']),
                np.array(payload['y_pred']),
                deep_metrics[model_name]['selected_threshold'],
            )
        )

if not prediction_summaries:
    raise FileNotFoundError(
        'No prediction artifacts found. Re-run the last sections of notebooks 03 and 04 before using this notebook.'
    )

# F1 is used here to sort operating-point performance after threshold selection.
threshold_metrics_df = pd.DataFrame(prediction_summaries).sort_values('F1', ascending=False).reset_index(drop=True)
display(threshold_metrics_df.style.format({
    'Threshold': '{:.2f}',
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'Specificity': '{:.4f}',
    'F1': '{:.4f}'
}))

threshold_metrics_df.to_csv(RESULTS_DIR / 'model_threshold_metrics.csv', index=False)
print('Saved threshold-based metrics to results/model_threshold_metrics.csv')

In [ ]:
# Show confusion matrices only for the strongest representative model from each
# family so the figure stays readable during the walkthrough.
confusion_payloads = []

if best_baseline_predictions_path.exists() and best_baseline_report_path.exists():
    confusion_payloads.append({
        'label': baseline_meta['best_model'],
        'family': 'Baseline ML',
        'y_true': baseline_preds['y_true'].to_numpy(),
        'y_pred': baseline_preds['y_pred_tuned'].to_numpy(),
    })

if deep_predictions_path.exists():
    # Choose the deep model by validation AUROC to stay consistent with the
    # model-selection logic used earlier in the project.
    best_deep_model = deep_df.sort_values('Val AUROC', ascending=False).iloc[0]['Model']
    best_deep_payload = deep_predictions[best_deep_model]
    confusion_payloads.append({
        'label': best_deep_model,
        'family': 'Deep Learning',
        'y_true': np.array(best_deep_payload['y_true']),
        'y_pred': np.array(best_deep_payload['y_pred']),
    })

fig, axes = plt.subplots(1, len(confusion_payloads), figsize=(7 * len(confusion_payloads), 5))
if len(confusion_payloads) == 1:
    axes = [axes]

for ax, payload in zip(axes, confusion_payloads):
    cm = confusion_matrix(payload['y_true'], payload['y_pred'])
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['No ADR', 'ADR'],
        yticklabels=['No ADR', 'ADR'],
        ax=ax,
    )
    ax.set_title(f"{payload['family']}\n{payload['label']}")
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'best_model_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

## Notes

- The combined ranking table covers every model saved by notebooks `03` and `04`.
- Threshold-based metrics depend on saved per-row predictions, so they currently include the best baseline model and each deep learning model.
- If you want threshold metrics for every baseline model, extend notebook `03` to save test predictions for all baselines, then re-run this notebook.